# Preserve generated CHASE-shape PNGs before a runtime reset

Copy this code cell into your **existing training notebook**, in the runtime that holds the generated images, after stopping its training cell. Opening this backup notebook may use a separate runtime without those images. Wait for `BACKUP COMPLETE` before disconnecting/deleting the runtime. It stores raw synthetic PNGs on Drive; weights and the training recipe are unchanged. PNG files are already compressed, so the archive needs roughly their existing total size.

On the next runtime, run the updated [main training notebook](https://colab.research.google.com/github/ikjoo5/solar_filament/blob/main/notebooks/SolarFilament_CHASE_Shape_T4.ipynb). It restores this archive and resumes from the last completed epoch. Integrity checks and reconstruction of other local caches still take time.


In [ ]:
# Stop the training cell first; keep this runtime connected until backup completes.
from pathlib import Path
from google.colab import drive
import hashlib, importlib.util, urllib.request
drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/filament-segmentation-2026/fixed_split_experiments/chase_shape_best84_recipe_t4_v1")
CACHE_HELPER_REV = "be7eee2b85bd1c99137e005f5bd870a541ab5363"
CACHE_HELPER_SHA = "3333cbe6b6a260c2ccb680a37eff69b36ef2b87e03122ee21613240d846de10c"
def cache_tools():
    target = OUTPUT / "cache_support" / (CACHE_HELPER_SHA + ".py")
    if target.is_symlink() or not target.resolve().is_relative_to(OUTPUT.resolve()):
        raise ValueError("Unsafe cache-helper path.")
    if target.is_file():
        raw = target.read_bytes()
    else:
        url = ("https://raw.githubusercontent.com/ikjoo5/solar_filament/" +
               CACHE_HELPER_REV + "/tools/chase_shape_cache.py")
        with urllib.request.urlopen(url, timeout=60) as response:
            raw = response.read(512001)
    if hashlib.sha256(raw).hexdigest() != CACHE_HELPER_SHA:
        raise ValueError("Cache-helper checksum differs; existing files were preserved.")
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        pending = target.with_suffix(".pending")
        try:
            pending.write_bytes(raw)
            pending.replace(target)
        finally:
            pending.unlink(missing_ok=True)
    spec = importlib.util.spec_from_file_location("solar_chase_cache", target)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
report = cache_tools().backup(OUTPUT)
print("Backup result:", report, flush=True)
if report["status"] not in ("backed_up", "already_backed_up"):
    raise RuntimeError("Backup did not complete; keep the runtime connected.")
print("BACKUP COMPLETE. You may now disconnect/delete this runtime.", flush=True)
